In [1]:
import os
%pwd
os.chdir("../")
from dataclasses import dataclass
from pathlib import Path
os.getcwd()

'/home/michauj/ds_e2e_project'

In [2]:
@dataclass
class ModelTrainingConfig:
    root_dir:Path
    train_data_path:Path
    test_data_path:Path
    model_name:str
    alpha:float
    l1_ratio:float
    random_state:int
    target_column:str

In [3]:
from src.ds_e2e_project.constants import *
from src.ds_e2e_project.utils.common import read_yaml, create_directories

In [4]:
class ConfigurationManager:
    def __init__(self,config_filepath=CONFIG_FILE_PATH,params_filepath = PARAMS_FILE_PATH,schema_filepath=SCHEMA_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])
        
    def get_model_training_config(self) -> ModelTrainingConfig:
        config = self.config.model_training
        params=self.params.ElasticNet
        schema=self.schema.TARGET_COLUMN
        create_directories([config.root_dir])

        model_training_config = ModelTrainingConfig(
        root_dir = config.root_dir,
        train_data_path = config.train_data_path,
        test_data_path = config.test_data_path,
        model_name = config.model_name,
        alpha = params.alpha,
        l1_ratio = params.l1_ratio,
        random_state = params.random_state,
        target_column = schema.name
        )

        return model_training_config

        

In [6]:
import pandas as pd
import os
from src.ds_e2e_project import logger
from sklearn.linear_model import ElasticNet
import joblib
class ModelTraining:
    def __init__(self, config: ModelTrainingConfig):
        self.config = config
    def train(self):
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

        train_x = train_data.drop([self.config.target_column],axis =1)
        test_x = test_data.drop([self.config.target_column], axis =1)
        train_y = train_data[[self.config.target_column]]
        test_y = test_data[[self.config.target_column]]

        lr = ElasticNet(
            alpha= self.config.alpha,
            l1_ratio= self.config.l1_ratio,
            random_state = self.config.random_state
        )
        lr.fit(train_x,train_y)

        joblib.dump(
            lr, os.path.join(self.config.root_dir, self.config.model_name)
        )

In [7]:
try:
    config = ConfigurationManager()
    model_training_config = config.get_model_training_config()
    model_training = ModelTraining(config = model_training_config)
    model_training.train()
except Exception as e:
    raise e

[2026-05-01 12:30:42,643: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-05-01 12:30:42,646: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-01 12:30:42,651: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-05-01 12:30:42,652: INFO: common: created directory at artifacts]
[2026-05-01 12:30:42,653: INFO: common: created directory at artifacts/model_training]


In [8]:
Path("config/config.yaml").exists()

True

In [9]:
!find ~/ds_e2e_project -name "config.yaml"

/home/michauj/ds_e2e_project/config/config.yaml


In [10]:
print(os.getcwd())

/home/michauj/ds_e2e_project
